In [1]:
!pip install torch torchvision torchaudio
!pip install torch-geometric
!pip install scikit-learn pandas
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.3.0-py3-none-any.whl (512 kB)


In [2]:
import torch
from torch_geometric.nn import GCNConv

print("All good")

All good


In [3]:
import os
import re
import random
import torch
import pandas as pd
import torch.nn.functional as F

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

In [4]:
base_path = r"C:\Users\Tanushree\Downloads\finegrained-traceability-main\finegrained-traceability-main\datasets\iTrust"

req_path = base_path + "/req"
code_path = base_path + "/code"

In [5]:
with open(base_path + "/all_req_filenames.txt") as f:
    req_files = [line.strip() for line in f]

with open(base_path + "/all_code_filenames.txt") as f:
    code_files = [line.strip() for line in f]

print(len(req_files), len(code_files))

131 226


In [6]:
with open(base_path + "/iTrust_solution_links.txt") as f:
    for i in range(10):
        print(repr(f.readline()))
        

'UC10E1.txt: GetUserNameAction.java\n'
'UC10E1.txt: AuthDAO.java\n'
'UC10E2.txt: GetUserNameAction.java\n'
'UC10E2.txt: AuthDAO.java\n'
'UC10S1.txt: EditPHRAction.java\n'
'UC10S1.txt: PersonnelDAO.java\n'
'UC10S1.txt: HealthRecordsDAO.java\n'
'UC10S1.txt: OfficeVisitDAO.java\n'
'UC10S1.txt: FamilyDAO.java\n'
'UC10S1.txt: EditHealthHistoryAction.java\n'


In [7]:
links = set()

for line in open(base_path + "/iTrust_solution_links.txt"):
    line = line.strip()
    
    if ":" not in line:
        continue
    
    req_part, code_part = line.split(":")

    req_name = req_part.strip()
    code_name = code_part.strip()

    # match requirement
    req_idx = None
    for i, r in enumerate(req_files):
        if req_name == r:
            req_idx = i
            break

    # match code
    code_idx = None
    for j, c in enumerate(code_files):
        if code_name == c:
            code_idx = j
            break

    if req_idx is not None and code_idx is not None:
        links.add((req_idx, code_idx))

print("Fixed links:", len(links))

Fixed links: 286


In [8]:
requirements = []

for file in req_files:
    with open(req_path + "/" + file, encoding="utf-8", errors="ignore") as f:
        requirements.append(f.read())

print("Requirements loaded:", len(requirements))

Requirements loaded: 131


In [9]:
codes = []
valid_code_files = []

for root, dirs, files in os.walk(code_path):
    for file in files:
        if file.endswith(".java") and file in code_files:
            try:
                with open(os.path.join(root, file), encoding="utf-8", errors="ignore") as f:
                    codes.append(f.read())
                    valid_code_files.append(file)
            except:
                pass

print("Code loaded:", len(codes))

Code loaded: 226


In [10]:
print(len(requirements))
print(len(codes))
print(len(links))

131
226
286


In [11]:
rows = []

# Positive samples
for (i, j) in links:
    rows.append([requirements[i], codes[j], 1])

# Negative samples (same number)
import random

neg_samples = []
while len(neg_samples) < len(rows):
    i = random.randint(0, len(requirements)-1)
    j = random.randint(0, len(codes)-1)

    if (i, j) not in links:
        neg_samples.append([requirements[i], codes[j], 0])

rows.extend(neg_samples)

df = pd.DataFrame(rows, columns=["requirement", "code", "label"])

print("Dataset size:", len(df))
print("Positive:", sum(df.label==1))
print("Negative:", sum(df.label==0))

Dataset size: 572
Positive: 286
Negative: 286


In [12]:
req_map = {i: requirements[i] for i in range(len(requirements))}
code_map = {i: codes[i] for i in range(len(codes))}

In [13]:
rows = []

# Positive samples
for (i, j) in links:
    if i < len(requirements) and j < len(codes):
        rows.append([requirements[i], codes[j], 1])

# Negative samples
import random

neg_samples = []
while len(neg_samples) < len(rows):
    i = random.randint(0, len(requirements)-1)
    j = random.randint(0, len(codes)-1)

    if (i, j) not in links:
        neg_samples.append([requirements[i], codes[j], 0])

rows.extend(neg_samples)

df = pd.DataFrame(rows, columns=["requirement", "code", "label"])

print("Dataset size:", len(df))
print("Positive:", sum(df.label==1))
print("Negative:", sum(df.label==0))

Dataset size: 572
Positive: 286
Negative: 286


In [17]:
print(df.columns)

Index(['requirement', 'code', 'label'], dtype='object')


In [18]:
df.columns = ["req_text", "code_text", "label"]

In [21]:
print(train_df.columns)

Index(['req_text', 'code_text', 'label'], dtype='object')


In [22]:
from sklearn.model_selection import train_test_split

train_df['req_text']
test_df['req_text']

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 457
Test: 115


In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec_req = TfidfVectorizer(max_features=200)
vec_code = TfidfVectorizer(max_features=200)

req_train = vec_req.fit_transform(train_df['requirement']).toarray()
code_train = vec_code.fit_transform(train_df['code']).toarray()

req_test = vec_req.transform(test_df['requirement']).toarray()
code_test = vec_code.transform(test_df['code']).toarray()

KeyError: 'requirement'

In [24]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

req_embeddings = model.encode(df["req_text"].tolist())
code_embeddings = model.encode(df["code_text"].tolist())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
import torch
import numpy as np

features = torch.tensor(
    np.vstack((req_embeddings, code_embeddings)),
    dtype=torch.float
)

NameError: name 'req_embeddings' is not defined

In [17]:
import numpy as np

test_features = torch.tensor(
    np.vstack((req_test, code_test)),
    dtype=torch.float
)

edges = []
train_size = len(train_df)
for i in range(train_size):
    edges.append([i, i + train_size])
    edges.append([i + train_size, i])

edge_index = torch.tensor(edges).t().contiguous()

train_data = Data(x=features, edge_index=edge_index)

NameError: name 'features' is not defined

In [ ]:
test_size = len(test_df)

edges_test = []

for i in range(test_size):
    edges_test.append([i, i + test_size])
    edges_test.append([i + test_size, i])

edge_index_test = torch.tensor(edges_test).t().contiguous()

test_data = Data(x=test_features, edge_index=edge_index_test)

In [ ]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

class GCN(torch.nn.Module):
    def __init__(self, in_c, hid):
        super().__init__()
        self.conv1 = GCNConv(in_c, hid)
        self.conv2 = GCNConv(hid, hid)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, in_c, hid):
        super().__init__()
        self.conv1 = GCNConv(in_c, hid)
        self.conv2 = GCNConv(hid, hid)
        self.classifier = torch.nn.Linear(hid * 2, 1)

    def forward(self, x, edge_index, pairs):
        x = F.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)

        # combine node pairs
        x_i = x[pairs[:, 0]]
        x_j = x[pairs[:, 1]]

        x_pair = torch.cat([x_i, x_j], dim=1)

        return torch.sigmoid(self.classifier(x_pair))

In [ ]:
model = GCN(features.shape[1], 16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
criterion = torch.nn.BCELoss()

def train():
    model.train()
    optimizer.zero_grad()

    pred = model(train_data.x, train_data.edge_index, pairs)
    loss = criterion(pred, labels)

    loss.backward()
    optimizer.step()

    return loss.item()

for epoch in range(50):
    print("Epoch", epoch, "Loss:", train())

In [ ]:
test_pairs = []

for i in range(test_size):
    test_pairs.append([i, i + test_size])

test_pairs = torch.tensor(test_pairs)

In [ ]:
test_size = len(test_df)

test_features = torch.tensor(
    list(req_test) + list(code_test),
    dtype=torch.float
)

edges_test = []

for i in range(test_size):
    edges_test.append([i, i + test_size])
    edges_test.append([i + test_size, i])

edge_index_test = torch.tensor(edges_test).t().contiguous()

test_data = Data(x=test_features, edge_index=edge_index_test)

model.eval()

pred = model(test_data.x, test_data.edge_index, test_pairs)

preds = (pred.detach().numpy() > 0.5).astype(int).flatten()
actual = test_df['label'].tolist()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:", accuracy_score(actual, preds))
print("Precision:", precision_score(actual, preds))
print("Recall:", recall_score(actual, preds))
print("F1:", f1_score(actual, preds))